In [ ]:
from archetype_style_analysis import *
from archetype_visualisation import *
from preprocessing import *

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# We use vgg19_bn instead of vgg19 since it is better
model = models.vgg19_bn(weights=models.VGG19_BN_Weights)
model = model.to(device)
model.eval()
return_nodes = {
    "features.2": "layer1",
    "features.5": "layer2",
    "features.9": "layer3",
    "features.12": "layer4",
    "features.16": "layer5",
}
feature_extractor = create_feature_extractor(model, return_nodes=return_nodes)

In [ ]:
all_paths = glob("ArtemisArt/**/*.jpg")
paths = all_paths[::2]
print(f"Il y a {len(paths)} images dans le dataset")

In [ ]:
def imshow_filter(kernels, row, col):
    print("-------------------------------------------------------------")
    plt.figure(figsize=(10, 10))
    for i in range(len(kernels)):
        img = np.mean(kernels[i], axis=0)
        img = img - img.min()
        img = img / img.max()

        plt.subplot(row, col, i + 1)
        plt.imshow(img, cmap="gray")
        plt.xticks([])
        plt.yticks([])
    plt.show()


conv_layers = []
for name, module in model.named_modules():
    if type(module).__name__ == "Conv2d":
        conv_layers.append(module)

### !!!! Attention, y'a vraiment beaucoup d'images qui sont générés donc vaut mieux
### sélectionner un layer en particulier !!!!

for conv in conv_layers:
    print(conv)
    kernels = conv.weight.data.cpu().numpy()
    print(kernels.shape)
    rows = 8
    imshow_filter(kernels, rows, kernels.shape[0] // rows)

In [ ]:
def visualize_feature_maps(image_path, extractor, layer_name="layer4", max_features=64):
    print("-" * 60)
    img = imread_safe(image_path)

    preprocess_img = preprocessing_image(img)
    tensor_img = preprocess_img.to(device)

    with torch.no_grad():
        features_dict = extractor(tensor_img)

    f_map = features_dict[layer_name].squeeze(0).cpu().numpy()

    print(f"Layer name : {layer_name}")

    n_features = min(f_map.shape[0], max_features)
    col = 8
    row = (n_features + col - 1) // col

    plt.figure(figsize=(15, 2 * row))

    for i in range(n_features):
        activation = f_map[i]

        activation = activation - activation.min()
        if activation.max() != 0:
            activation = activation / activation.max()

        plt.subplot(row, col, i + 1)
        plt.imshow(activation, cmap="gray")
        plt.xticks([])
        plt.yticks([])

    plt.tight_layout()
    plt.show()
    plt.close()


img_path = paths[0]
print(img_path)

visualize_feature_maps(img_path, feature_extractor, layer_name="layer1")
visualize_feature_maps(img_path, feature_extractor, layer_name="layer2")
visualize_feature_maps(img_path, feature_extractor, layer_name="layer3")
visualize_feature_maps(img_path, feature_extractor, layer_name="layer4")
visualize_feature_maps(img_path, feature_extractor, layer_name="layer5")

In [ ]:
n_archetype = 8
a = ArchetypeGenerator(n_archetype, paths, device, feature_extractor)
A, B, Z, ipca_mod = a.find_archetypes()

In [ ]:
image_synthetisee = synthetiser_archetype(
    Z_archetype=Z_arch_0,
    pca_model=pca_mod,
    extractor=feature_extractor,
    device=device,
    n_iteration=500,
)
plt.imshow(image_synthetisee)
plt.axis("off")
plt.title("L'essence de l'Archétype 0")
plt.savefig("archetype_0.png", bbox_inches="tight", dpi=300)
print("Image sauvegardée sous le nom 'archetype_0.png' !")

plt.close()